# Enron lexical retrieval: Table 2 dictionary, window rules and TF-IDF

The lexical channel of the retrieval design (proposal Section 4.2). It returns a ranked top 1,000 for reciprocal rank fusion with the semantic channel. Emails are ranked by **rule match first** (a Table 2 combination inside one 200-word window), then **TF-IDF cosine similarity** to a fixed query, then `email_id`.

**Inputs.** `emails_analysis.parquet` from a verified run of `01_data_cleaning.ipynb` (under `OUTPUT_ROOT`); `data/structural_baseline.parquet` (which emails are searchable); `data/spe_subset_emails.csv` (recall check only).
**Outputs.** One timestamped folder per run under `OUTPUT_ROOT/lexical_outputs/`, holding the dictionary, the query, the settings, the top 1,000, and an audit of every rule match.

| Section | Content | Status |
|---|---|---|
| 1. Configuration | Paths, eligible set, cutoff, window sizes | Done |
| 2. Dictionary | Table 2 Groups A-D as regular expressions, with the EDA term classes | Done |
| 3. Query | Table 2 terms plus the Section 4.3 description, for TF-IDF | Done |
| 4. Matching and window rules | Positions of every match; rules R1 and R2 | Done |
| 5. Tests | Hand-made examples for every term and rule boundary | Done |
| 6-10 | Eligible set, rule firing counts, TF-IDF, ranking and outputs, checks | Added once the cleaning run is verified |

**Reading the results.** A rule match or a high TF-IDF score means an email uses the vocabulary of SPE risk transfer. It identifies leads for review, including ordinary guarantees, legitimate valuations and objections; it does not indicate wrongdoing.

## 1. Configuration
Paths must match `01_data_cleaning.ipynb`. The eligible-set settings are the proposed defaults; confirm them with the group (and the semantic channel, so both rank the same emails).

In [1]:
from pathlib import Path
from collections import Counter, namedtuple
from bisect import bisect_left, bisect_right
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import json
import re

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 220)

# --- Paths ---
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()   # notebooks run from notebooks/
OUTPUT_ROOT = Path('E:/University-Georgia Tech/INTA6450_dataanalytics_security/project')   # same as 01 and 02
CLEANING_RUNS = OUTPUT_ROOT / 'enron_cleaning_outputs'    # input: emails_analysis.parquet
LEXICAL_OUTPUTS = OUTPUT_ROOT / 'lexical_outputs'         # output: one timestamped folder per run
BASELINE_PATH = REPO_ROOT / 'data' / 'structural_baseline.parquet'
SPE_SUBSET_PATH = REPO_ROOT / 'data' / 'spe_subset_emails.csv'
HOUSTON = ZoneInfo('America/Chicago')

# --- Eligible set (proposed defaults; confirm with the group) ---
BASELINE_SET = 'primary'               # rank the primary baseline; fallback emails (no new text) are context only
APPLY_CUTOFF = True                    # proposal: rank emails dated before the cutoff
CUTOFF_LOCAL = '2001-10-16T00:00:00'   # Houston local time

# --- Rules ---
WINDOW_WORDS = 200    # proposal 4.2: rule terms must fall inside one window of this many words
CONTEXT_WORDS = 100   # EDA memo: context-required names count only with a Group B/C word this close

# --- Ranking ---
TOP_K = 1000
RANDOM_SEED = 6450

# Known emails from the proposal. They shaped the dictionary, so finding them checks the pipeline, not its accuracy.
SEED_EMAILS = {
    'c9732ecec3d1333cbf8ce9d461b7fe7f': 'Kaminski, "LJM/Raptor valuations", 2001-10-04',
    'f6a682d86455b20afb09bfde1adf2b76': 'Pruett, "CGAS Hawaii Sec 125 Trust", 2001-05-02',
}


def cutoff_utc(houston_local):
    """Houston local time without an offset -> UTC, daylight saving applied."""
    return datetime.fromisoformat(houston_local).replace(tzinfo=HOUSTON).astimezone(timezone.utc)


assert cutoff_utc('2001-10-16T00:00:00') == datetime(2001, 10, 16, 5, tzinfo=timezone.utc)   # CDT is UTC-5
CUTOFF_UTC = cutoff_utc(CUTOFF_LOCAL) if APPLY_CUTOFF else None
for required in [OUTPUT_ROOT, BASELINE_PATH, SPE_SUBSET_PATH]:
    if not required.exists():
        raise FileNotFoundError(f'Missing {required}; check the paths above.')
print('Repo:', REPO_ROOT)
print('Outputs:', OUTPUT_ROOT)
print(f'Rank the {BASELINE_SET} baseline; cutoff {CUTOFF_LOCAL} Houston = {CUTOFF_UTC}' if APPLY_CUTOFF
      else f'Rank the {BASELINE_SET} baseline; no cutoff')

Repo: C:\Users\jiawe\Documents\GitHub\Enron
Outputs: E:\University-Georgia Tech\INTA6450_dataanalytics_security\project
Rank the primary baseline; cutoff 2001-10-16T00:00:00 Houston = 2001-10-16 05:00:00+00:00


## 2. Dictionary (proposal Table 2)

Four groups: **A** entities and transaction labels, **B** retained exposure and protection, **C** accounting and representation, **D** action, awareness and pressure. Matching ignores case, allows a space or hyphen inside compound phrases (and a line break, since email bodies wrap), recognizes grammatical variants, and requires word boundaries.

**Group A classes** come from the EDA precision review (`docs/eda_decision_memo.md`):
- `standalone`: counts wherever it appears.
- `context`: counts only with a Group B or C word within `CONTEXT_WORDS` words. The proposal requires this for "single ambiguous words such as Hawaii or Marlin"; the EDA found eight such names.
- `dropped`: matched and logged for the audit, but never counts. Bare "Hawaii" is a place name in about 80% of emails; "Hawaii 125" stays as its own stand-alone term.

**Departures from the proposal, to report in the paper:**
1. Context gating and the dropped class above (EDA evidence, not in Table 2).
2. Timberwolf, Bobcat, Southampton and Cuiaba are EDA candidate names, not Table 2 terms. They need Powers report page citations before the paper relies on them.
3. R2 ("B plus C plus D without a named entity") is read as: a window holding B, C and D terms, with no entity required in that window. An email can match R1 and R2 at once.

In [2]:
# group, label, regular expression, class, source
DICTIONARY_ROWS = [
    # --- A. Entities and transaction labels (patterns match 02_eda.ipynb, so the SPE subset is comparable) ---
    ('A', 'raptor',           r'\braptors?\b',                                        'standalone', 'Table 2; covers Raptor I-IV and 1-4'),
    ('A', 'ljm',              r'\bljm\d?\b',                                          'standalone', 'Table 2; covers LJM1, LJM2, LJM Cayman, LJM Co-Investment'),
    ('A', 'chewco',           r'\bchewco\b',                                          'standalone', 'Table 2'),
    ('A', 'jedi',             r'\bjedi\b',                                            'standalone', 'Table 2; covers JEDI II'),
    ('A', 'hawaii_125',       r'\bhawaii[ -]?(?:sec(?:tion|\.)?[ -]?)?125(?:-0)?\b',   'standalone', 'Table 2 (Hawaii 125-0, Hawaii Sec 125)'),
    ('A', 'whitewing',        r'\bwhitewing\b',                                       'standalone', 'Table 2'),
    ('A', 'condor',           r'\bcondor\b',                                          'standalone', 'Table 2'),
    ('A', 'trutta',           r'\btrutta\b',                                          'standalone', 'Table 2'),
    ('A', 'project_velocity', r'\bproject\s+velocity\b',                              'standalone', 'Table 2'),
    ('A', 'talon',            r'\btalon\b',                                           'context',    'Table 2'),
    ('A', 'porcupine',        r'\bporcupine\b',                                       'context',    'Table 2'),
    ('A', 'osprey',           r'\bosprey\b',                                          'context',    'Table 2'),
    ('A', 'marlin',           r'\bmarlin\b',                                          'context',    'Table 2'),
    ('A', 'rhythms',          r'\brhythms\b',                                         'context',    'Table 2'),
    ('A', 'timberwolf',       r'\btimber[\s-]?wolf\b',                                'context',    'EDA candidate; needs Powers page citation'),
    ('A', 'bobcat',           r'\bbobcat\b',                                          'context',    'EDA candidate; needs Powers page citation'),
    ('A', 'southampton',      r'\bsouthampton\b',                                     'context',    'EDA candidate; needs Powers page citation'),
    ('A', 'hawaii',           r'\bhawaii\b',                                          'dropped',    'Table 2; dropped as a stand-alone term (EDA precision 20%)'),
    ('A', 'cuiaba',           r'\bcuiab[aá]',                                         'dropped',    'EDA candidate; dropped (EDA precision 0-20%)'),

    # --- B. Retained exposure and protection ---
    ('B', 'guarantee',             r'\bguarant(?:y|ies|ied|ying|ee|ees|eed|eeing)\b',         '-', 'Table 2: guarantee/guaranty'),
    ('B', 'recourse',              r'\brecourse\b',                                          '-', 'Table 2'),
    ('B', 'indemnity',             r'\bindemni(?:ty|ties|fy|fies|fied|fying|fication|fications)\b', '-', 'Table 2'),
    ('B', 'make_whole',            r'\bmake[\s-]?whole\b',                                   '-', 'Table 2: make whole/make-whole'),
    ('B', 'repurchase',            r'\brepurchas(?:e|es|ed|ing)\b',                          '-', 'Table 2'),
    ('B', 'buyback',               r'\b(?:buy(?:s|ing)?|bought)[\s-]?backs?\b',              '-', 'Table 2: buyback/buy back'),
    ('B', 'total_return_swap',     r'\btotal[\s-]+return[\s-]+swaps?\b',                     '-', 'Table 2'),
    ('B', 'contingent_obligation', r'\bcontingent\s+obligations?\b',                         '-', 'Table 2'),
    ('B', 'credit_support',        r'\bcredit\s+support\b',                                  '-', 'Table 2'),
    ('B', 'credit_capacity',       r'\bcredit\s+capacity\b',                                 '-', 'Table 2'),
    ('B', 'collateral',            r'\bcollateral(?:s|ized|ize|izes|izing|ised|ise|ization|isation)?\b', '-', 'Table 2'),
    ('B', 'enron_stock',           r'\b(?:enron|ene)\s+stock\b',                             '-', 'Table 2: Enron stock; ENE stock'),
    ('B', 'equity_at_risk',        r'\bequity\s+at\s+risk\b',                                '-', 'Table 2'),
    ('B', 'side_agreement',        r'\bside[\s-]+(?:agreements?|letters?)\b',                '-', 'Table 2: side agreement; side letter'),

    # --- C. Accounting and representation ---
    ('C', 'off_balance_sheet',       r'\boff[\s-]+(?:the[\s-]+)?balance[\s-]+sheet\b',        '-', 'Table 2: off-balance-sheet/off balance sheet'),
    ('C', 'consolidation',           r'\bconsolidat(?:e|es|ed|ing|ion|ions)\b',              '-', 'Table 2'),
    ('C', 'deconsolidation',         r'\bdeconsolidat(?:e|es|ed|ing|ion|ions)\b',            '-', 'Table 2'),
    ('C', 'hedge',                   r'\bhedg(?:e|es|ed|ing)\b',                             '-', 'Table 2'),
    ('C', 'accounting_hedge',        r'\baccounting\s+hedges?\b',                            '-', 'Table 2'),
    ('C', 'mark_to_market',          r'\bmark(?:s|ed|ing)?[\s-]+to[\s-]+market\b|\bmtm\b',    '-', 'Table 2: mark-to-market/MTM'),
    ('C', 'valuation',               r'\bvaluations?\b',                                     '-', 'Table 2'),
    ('C', 'write_down',              r'\b(?:write|writes|writing|written|wrote)[\s-]?downs?\b', '-', 'Table 2: write-down/write down'),
    ('C', 'impairment',              r'\bimpair(?:ment|ments|ed)\b',                         '-', 'Table 2'),
    ('C', 'credit_reserve',          r'\bcredit\s+reserves?\b',                              '-', 'Table 2'),
    ('C', 'loss_recognition',        r'\bloss\s+recognition\b',                              '-', 'Table 2'),
    ('C', 'disclosure',              r'\bdisclosures?\b',                                    '-', 'Table 2'),
    ('C', 'footnote',                r'\bfoot[\s-]?notes?\b',                                '-', 'Table 2'),
    ('C', 'auditor',                 r'\bauditors?\b',                                       '-', 'Table 2'),
    ('C', 'independent_third_party', r'\bindependent\s+third[\s-]+part(?:y|ies)\b',          '-', 'Table 2'),

    # --- D. Action, awareness and pressure ---
    ('D', 'backdate',          r'\bback[\s-]?dat(?:e|es|ed|ing)\b',                    '-', 'Table 2: backdate/backdated'),
    ('D', 'effective_as_of',   r'\beffective\s+as\s+of\b',                             '-', 'Table 2'),
    ('D', 'cannot_support',    r"\b(?:cannot|can\s+not|can['’]t)\s+support\b",         '-', 'Table 2'),
    ('D', 'insufficient',      r'\binsufficien(?:t|cy)\b',                             '-', 'Table 2'),
    ('D', 'shortfall',         r'\bshort[\s-]?falls?\b',                               '-', 'Table 2'),
    ('D', 'no_risk_transfer',  r'\bno\s+risk\s+transfer\b',                            '-', 'Table 2'),
    ('D', 'retain_the_mark',   r'\bretain(?:s|ed|ing)?\s+the\s+mark\b',                '-', 'Table 2'),
    ('D', 'defer_recognition', r'\bdefer(?:s|red|ring)?\s+(?:the\s+)?recognition\b',   '-', 'Table 2'),
    ('D', 'do_not_disclose',   r"\b(?:do\s+not|don['’]t|not\s+to)\s+disclose\b",       '-', 'Table 2'),
    ('D', 'leave_out',         r'\b(?:leave|leaves|leaving|left)\s+out\b',             '-', 'Table 2'),
    ('D', 'omit',              r'\bomi(?:t|ts|tted|tting|ssion|ssions)\b',             '-', 'Table 2'),
    ('D', 'sign_off',          r'\bsign(?:s|ed|ing)?[\s-]?offs?\b',                    '-', 'Table 2'),
    ('D', 'quarter_end',       r'\bquarter[\s-]?end\b',                                '-', 'Table 2'),
    ('D', 'earnings_target',   r'\bearnings\s+targets?\b',                             '-', 'Table 2'),
]
DICTIONARY = pd.DataFrame(DICTIONARY_ROWS, columns=['group', 'label', 'pattern', 'term_class', 'source'])

assert DICTIONARY['label'].is_unique, 'labels must be unique'
assert set(DICTIONARY['group']) == {'A', 'B', 'C', 'D'}
assert set(DICTIONARY.loc[DICTIONARY['group'] == 'A', 'term_class']) == {'standalone', 'context', 'dropped'}
assert set(DICTIONARY.loc[DICTIONARY['group'] != 'A', 'term_class']) == {'-'}

PATTERNS = [(group, label, term_class, re.compile(pattern, re.IGNORECASE))
            for group, label, pattern, term_class in zip(DICTIONARY['group'], DICTIONARY['label'],
                                                         DICTIONARY['pattern'], DICTIONARY['term_class'])]
display(DICTIONARY.groupby(['group', 'term_class']).size().rename('terms').reset_index())

,group,term_class,terms
0,A,context,8
1,A,dropped,2
2,A,standalone,9
3,B,-,14
4,C,-,15
5,D,-,14


## 3. Query for TF-IDF (proposal Section 4.2)

"Create one lexical query by concatenating all Table 2 terms once with the semantic description in Section 4.3." The terms are copied as printed in Table 2 (so the EDA candidate names are not in the query), with `/` read as a separator between alternatives. The description is copied verbatim from Section 4.3.

In [3]:
TABLE2_TERMS = {
    'A': ['Raptor', 'Raptors', 'Raptor I/II/III/IV and 1/2/3/4', 'Talon', 'Porcupine', 'LJM', 'LJM1', 'LJM2', 'LJM Cayman',
          'LJM Co-Investment', 'Chewco', 'JEDI', 'JEDI II', 'Hawaii', 'Hawaii 125-0', 'Hawaii Sec 125', 'Whitewing', 'Condor',
          'Osprey', 'Marlin', 'Project Velocity', 'Trutta', 'Rhythms'],
    'B': ['guarantee/guaranty', 'recourse', 'indemnity', 'make whole/make-whole', 'repurchase', 'buyback/buy back',
          'total return swap', 'contingent obligation', 'credit support', 'credit capacity', 'collateral', 'Enron stock',
          'ENE stock', 'equity at risk', 'side agreement', 'side letter'],
    'C': ['off-balance-sheet/off balance sheet', 'consolidation', 'deconsolidation', 'hedge', 'accounting hedge',
          'mark-to-market/MTM', 'valuation', 'write-down/write down', 'impairment', 'credit reserve', 'loss recognition',
          'disclosure', 'footnote', 'auditor', 'independent third party'],
    'D': ['backdate/backdated', 'effective as of', 'cannot support', 'insufficient', 'shortfall', 'no risk transfer',
          'retain the mark', 'defer recognition', 'do not disclose', 'leave out', 'omit', 'sign off', 'quarter-end',
          'earnings target'],
}

SEMANTIC_DESCRIPTION = (
    'Internal business correspondence about a special-purpose entity, related-party asset transfer, or purported hedge '
    'in which Enron remains responsible for material losses through guarantees, repurchase promises, total-return swaps, '
    'collateral commitments, or support based on its own shares. Participants acknowledge that an outside party lacks '
    'independent capital, that protection is insufficient, or that an asset has lost value. The correspondence then '
    'discusses presenting the risk as transferred, preserving an unsupported accounting benefit, postponing recognition '
    'of the known loss, changing a transaction date or valuation basis after the loss, or withholding a material '
    'obligation from auditors or decision-makers. Relevant evidence connects the retained exposure to a participant\'s '
    'awareness and a misleading action involving the same transaction. Communications may express this indirectly '
    'through requests, approvals, revisions, or objections rather than explicitly using the word fraud.'
)

LEXICAL_QUERY = ' '.join(term.replace('/', ' ') for group in 'ABCD' for term in TABLE2_TERMS[group]) + '\n' + SEMANTIC_DESCRIPTION
assert sum(len(v) for v in TABLE2_TERMS.values()) == 68
print(f'{sum(len(v) for v in TABLE2_TERMS.values())} Table 2 terms + description: {len(LEXICAL_QUERY.split())} words')
print(LEXICAL_QUERY[:300], '...')

68 Table 2 terms + description: 256 words
Raptor Raptors Raptor I II III IV and 1 2 3 4 Talon Porcupine LJM LJM1 LJM2 LJM Cayman LJM Co-Investment Chewco JEDI JEDI II Hawaii Hawaii 125-0 Hawaii Sec 125 Whitewing Condor Osprey Marlin Project Velocity Trutta Rhythms guarantee guaranty recourse indemnity make whole make-whole repurchase buybac ...


## 4. Matching and window rules

`find_hits` records every dictionary match with its character span and **word index** (words are runs of non-space characters), so window distances are measured in words as the proposal specifies, and every match position can be saved for the audit log.

`apply_rules` decides, for one email:
- **Qualifying names**: Group A hits that count. Stand-alone names always count; context names need a Group B/C hit within `CONTEXT_WORDS` words; dropped names never count.
- **R1**: a qualifying name and any B, C or D hit inside one `WINDOW_WORDS`-word window (at most 199 words apart).
- **R2**: a B hit, a C hit and a D hit inside one window, with no name required.

It also returns the word span of the first window that fired, for snippets.

In [4]:
Hit = namedtuple('Hit', 'group label term_class start end word')
WORD = re.compile(r'\S+')


def find_hits(text):
    """Every dictionary match in `text`, sorted by position, with character span and word index."""
    found = [(group, label, term_class, m.start(), m.end())
             for group, label, term_class, regex in PATTERNS for m in regex.finditer(text)]
    if not found:
        return []
    word_starts = [m.start() for m in WORD.finditer(text)]
    hits = [Hit(g, l, c, s, e, max(bisect_right(word_starts, s) - 1, 0)) for g, l, c, s, e in found]
    return sorted(hits, key=lambda h: (h.start, h.label))


def _near(words, position, distance):
    """True if any value in the sorted list `words` is within `distance` of `position`."""
    i = bisect_left(words, position - distance)
    return i < len(words) and words[i] <= position + distance


def apply_rules(hits, window=WINDOW_WORDS, context=CONTEXT_WORDS):
    """Rule decision for one email's hits (as returned by find_hits)."""
    reach = window - 1                                   # two words share a `window`-word window if <= window-1 apart
    bc_words = [h.word for h in hits if h.group in ('B', 'C')]
    bcd = [h for h in hits if h.group in ('B', 'C', 'D')]
    bcd_words = [h.word for h in bcd]

    names = [h for h in hits if h.group == 'A' and (h.term_class == 'standalone'
                                                    or (h.term_class == 'context' and _near(bc_words, h.word, context)))]

    # R1: a qualifying name with any B/C/D hit inside one window
    r1_span = None
    for a in names:
        i = bisect_left(bcd_words, a.word - reach)
        if i < len(bcd_words) and bcd_words[i] <= a.word + reach:
            r1_span = (min(a.word, bcd_words[i]), max(a.word, bcd_words[i]))
            break

    # R2: B, C and D inside one window (sliding window over B/C/D hits in word order)
    r2_span, counts, left = None, Counter(), 0
    for right, hit in enumerate(bcd):
        counts[hit.group] += 1
        while hit.word - bcd[left].word > reach:
            counts[bcd[left].group] -= 1
            left += 1
        if counts['B'] and counts['C'] and counts['D']:
            r2_span = (bcd[left].word, hit.word)
            break

    return {
        'names': sorted({h.label for h in names}),
        'groups': ''.join(sorted({h.group for h in hits if h.term_class != 'dropped'})),
        'r1': r1_span is not None, 'r2': r2_span is not None,
        'r1_span': r1_span, 'r2_span': r2_span,
    }


def labels_in(text):
    return {h.label for h in find_hits(text)}

## 5. Tests
Hand-made examples, no data needed. Every dictionary term has at least one positive example, the main variants are covered, near-misses must not match, and the window and context boundaries are tested exactly at the limit. If you edit the dictionary, add an example here.

In [5]:
POSITIVE = {
    # A
    'raptor': ['Raptor III', 'the RAPTORS', 'LJM/Raptor valuations'], 'ljm': ['LJM2 partners', 'LJM Cayman', 'LJM Co-Investment'],
    'chewco': ['Chewco'], 'jedi': ['JEDI II'], 'hawaii_125': ['Hawaii 125-0 Trust', 'CGAS Hawaii Sec 125 Trust', 'Hawaii Sec. 125'],
    'whitewing': ['Whitewing'], 'condor': ['Condor'], 'trutta': ['Trutta'], 'project_velocity': ['Project\nVelocity'],
    'talon': ['Talon'], 'porcupine': ['Porcupine'], 'osprey': ['Osprey Trust'], 'marlin': ['Marlin'], 'rhythms': ['Rhythms'],
    'timberwolf': ['Timber-wolf', 'timberwolf', 'Timber Wolf'], 'bobcat': ['Bobcat'], 'southampton': ['Southampton'],
    'hawaii': ['Hawaii'], 'cuiaba': ['Cuiabá', 'Cuiaba'],
    # B
    'guarantee': ['guaranty', 'guaranteed', 'GUARANTEES', 'guaranties'], 'recourse': ['non-recourse debt'],
    'indemnity': ['indemnification', 'indemnify'], 'make_whole': ['make-whole', 'make whole'], 'repurchase': ['repurchased'],
    'buyback': ['buyback', 'buy back', 'bought back', 'buy-backs'], 'total_return_swap': ['total-return swap', 'total return\nswaps'],
    'contingent_obligation': ['contingent obligations'], 'credit_support': ['credit support'], 'credit_capacity': ['credit capacity'],
    'collateral': ['collateral', 'collateralized'], 'enron_stock': ['Enron stock', 'ENE stock'], 'equity_at_risk': ['equity at risk'],
    'side_agreement': ['side letter', 'side-agreement'],
    # C
    'off_balance_sheet': ['off-balance-sheet', 'off balance sheet', 'off-balance sheet', 'off the balance sheet'],
    'consolidation': ['consolidate', 'consolidation'], 'deconsolidation': ['deconsolidated'], 'hedge': ['hedging', 'hedged'],
    'accounting_hedge': ['accounting hedges'], 'mark_to_market': ['mark-to-market', 'MTM', 'marked to market'],
    'valuation': ['valuations'], 'write_down': ['write-down', 'written down', 'writedowns'], 'impairment': ['impairment', 'impaired'],
    'credit_reserve': ['credit reserves'], 'loss_recognition': ['loss recognition'], 'disclosure': ['disclosures'],
    'footnote': ['foot-note', 'footnotes'], 'auditor': ['auditors'], 'independent_third_party': ['independent third-party'],
    # D
    'backdate': ['backdated', 'back-dated', 'back dating'], 'effective_as_of': ['effective as\nof'],
    'cannot_support': ['cannot support', 'can not support', "can't support", 'can’t support'], 'insufficient': ['insufficiency'],
    'shortfall': ['short-fall', 'shortfalls'], 'no_risk_transfer': ['no risk transfer'], 'retain_the_mark': ['retaining the mark'],
    'defer_recognition': ['deferred recognition', 'defer the recognition'],
    'do_not_disclose': ["don't disclose", 'do not disclose', 'not to disclose'], 'leave_out': ['left out'],
    'omit': ['omitted', 'omission'], 'sign_off': ['signed off', 'sign-off', 'signoff'],
    'quarter_end': ['quarter end', 'quarter-end', 'quarterend'], 'earnings_target': ['earnings targets'],
}
NEGATIVE = {
    'hedge': ['hedgehog'], 'auditor': ['the auditorium'], 'omit': ['omelette'], 'raptor': ['raptorial'],
    'consolidation': ['deconsolidation'], 'ljm': ['ljmx'], 'hawaii_125': ['Hawaii 1250'], 'disclosure': ['disclosed'],
    'cannot_support': ['can support'], 'mark_to_market': ['market to market'], 'valuation': ['devaluation'],
}
assert set(POSITIVE) == set(DICTIONARY['label']), f'terms without a test: {set(DICTIONARY["label"]) - set(POSITIVE)}'
for label, texts in POSITIVE.items():
    for text in texts:
        assert label in labels_in(text), (label, text)
for label, texts in NEGATIVE.items():
    for text in texts:
        assert label not in labels_in(text), (label, text)

# Word positions
assert [h.word for h in find_hits('a b LJM c hedge')] == [2, 4]
assert find_hits('LJM/Raptor')[0].word == find_hits('LJM/Raptor')[1].word == 0   # two names inside one token

filler = lambda n: ' '.join(['x'] * n)
rules = lambda text: apply_rules(find_hits(text))

# R1 boundary: name at word 0, partner at word 199 (same window) vs word 200 (outside)
assert rules('Raptor ' + filler(198) + ' valuation')['r1']
assert rules('Raptor ' + filler(198) + ' valuation')['r1_span'] == (0, 199)
assert not rules('Raptor ' + filler(199) + ' valuation')['r1']
assert rules('backdated ' + filler(150) + ' LJM')['r1']                     # D partner, partner before the name

# Context names: need a B/C word within CONTEXT_WORDS; D does not count as context
assert rules('Talon ' + filler(99) + ' hedge')['names'] == ['talon']
assert rules('Talon ' + filler(100) + ' hedge')['names'] == []
assert not rules('Talon ' + filler(100) + ' hedge')['r1']
assert rules('Talon sign off')['names'] == [] and not rules('Talon sign off')['r1']

# Dropped names never count; Hawaii 125 does
assert rules('Hawaii vacation; we should hedge the loss')['names'] == []
assert not rules('Hawaii vacation; we should hedge the loss')['r1']
assert rules('Hawaii 125-0 guarantee')['names'] == ['hawaii_125'] and rules('Hawaii 125-0 guarantee')['r1']

# R2: B + C + D in one window, no name required
assert rules('guarantee ' + filler(50) + ' hedge ' + filler(50) + ' backdated')['r2']
assert not rules('guarantee ' + filler(50) + ' hedge ' + filler(50) + ' backdated')['r1']
assert not rules('guarantee hedge ' + filler(250) + ' backdated')['r2']
assert not rules('guarantee hedge valuation')['r2']                         # no D term
assert rules('guarantee ' + filler(197) + ' hedge backdated')['r2']         # words 0, 198, 199
assert not rules('guarantee ' + filler(198) + ' hedge backdated')['r2']     # words 0, 199, 200
far = rules('Raptor ' + filler(300) + ' guarantee hedge backdated')         # name present but far away
assert far['r2'] and not far['r1'] and far['names'] == ['raptor']

# Seed-like text (Kaminski, Oct 4, 2001): two names plus valuation and "cannot support"
seed = rules('ljm/raptor valuations\nI feel strongly that I cannot support the valuations my group has produced')
assert seed['r1'] and seed['names'] == ['ljm', 'raptor'] and seed['groups'] == 'ACD'

assert rules('Lunch on Friday at noon.') == {'names': [], 'groups': '', 'r1': False, 'r2': False, 'r1_span': None, 'r2_span': None}
print(f'Dictionary and rule tests passed: {sum(map(len, POSITIVE.values()))} positive and '
      f'{sum(map(len, NEGATIVE.values()))} negative examples, {len(DICTIONARY)} terms.')

Dictionary and rule tests passed: 108 positive and 11 negative examples, 62 terms.
